In [ ]:
%%capture
!pip install --proxy=192.168.2.10:8080 --upgrade snowflake-connector-python[pandas]
!pip install --proxy=192.168.2.10:8080 --upgrade keyring seaborn statsmodels

In [ ]:
import os
import json
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import snowflake.connector

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

import warnings
warnings.filterwarnings('ignore')

# Setting Up the Connector

In [ ]:
# load the credentials
with open('./credentials.json', 'r') as config_file:
    config = json.load(config_file)

snowflake_creds = config['snowflake_credentials']

In [ ]:
ctx = snowflake.connector.connect(**snowflake_creds)
cur = ctx.cursor()

# Load Datasets

In [ ]:
cur.execute("""
SELECT * FROM IRONONETECH_DB.PUBLIC.V_FORECAST_CURVE_CONSOLIDATED;
""")
df = cur.fetch_pandas_all()
df.head()

# Data Preparation

In [ ]:
portfolio       = 'Buckeye'
sub_portfolio   = 'PP'
last_input_date = '2024-12-31'
TARGET_METRIC   = 'Ending Outstanding Loans'

In [ ]:
portfolio_df = df[df['PORTFOLIO'] == portfolio]
portfolio_df = portfolio_df[portfolio_df['SUB_PORTFOLIO'] == sub_portfolio]
portfolio_df = portfolio_df[portfolio_df['FORECAST_TYPE'] == 'Actual']

metrics = portfolio_df['METRIC'].unique()
print('Available metrics:', metrics)

In [ ]:
dff = portfolio_df.groupby(['DATE', 'METRIC']).sum().reset_index()

data_df = dff.pivot(
    index='DATE',
    columns='METRIC',
    values='METRIC_VALUE'
).sort_index()

train_df = data_df[data_df.index <= datetime.date.fromisoformat(last_input_date)]
test_df  = data_df[data_df.index >  datetime.date.fromisoformat(last_input_date)]

print(f"Train: {len(train_df)} rows  ({train_df.index[0]} → {train_df.index[-1]})")
print(f"Test : {len(test_df)} rows  ({test_df.index[0]} → {test_df.index[-1]})")
train_df.head()

# Feature Engineering — Seasonal Dummy Variables

ARIMAX uses two binary "switch" columns as exogenous variables:

- **`is_december`** — `1` when the month is December, `0` otherwise (captures the year-end spike)  
- **`is_june`** — `1` when the month is June, `0` otherwise (captures the mid-year dip)

This lets the model learn exactly two seasonal coefficients instead of 12, making it stable on ~20 data points.

In [ ]:
def make_dummies(index):
    """Create is_december and is_june dummy columns from a date index."""
    dates = pd.to_datetime(index)
    return pd.DataFrame({
        'is_december': (dates.month == 12).astype(int),
        'is_june'    : (dates.month == 6).astype(int),
    }, index=index)


exog_train = make_dummies(train_df.index)
exog_test  = make_dummies(test_df.index)

print("Training exog sample:")
display(exog_train.tail(6))
print("\nTest exog:")
display(exog_test)

# Modelling — ARIMAX(1,1,0)

An **ARIMAX(1,1,0)** model is fitted on the training series:

| Component | Value | Rationale |
|-----------|-------|-----------|
| p = 1 | AR(1) | PACF shows sharp cutoff after lag 1 |
| d = 1 | 1 difference | Removes the downward trend; series becomes stationary |
| q = 0 | No MA term | ACF does not indicate a moving-average process |

Two exogenous "switch" variables are included as regressors:
- **`is_december`** — captures the sharp year-end spike
- **`is_june`** — captures the mid-year dip

In [ ]:
train_y = train_df[TARGET_METRIC].astype(float)

arimax_model = SARIMAX(
    endog  = train_y,
    exog   = exog_train,
    order  = (1, 1, 0),
    trend  = 'n',
    enforce_stationarity  = False,
    enforce_invertibility = False,
)
result = arimax_model.fit(disp=False)

print(result.summary())

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# ── ARIMAX out-of-sample forecast ────────────────────────────────────────────
forecast_obj  = result.get_forecast(steps=len(test_df), exog=exog_test)
forecast_mean = forecast_obj.predicted_mean

# ── Pull 2025 0+12 benchmark from raw Snowflake data ─────────────────────────
reforecast_mask = (
    (df['PORTFOLIO']     == portfolio) &
    (df['SUB_PORTFOLIO'] == sub_portfolio) &
    (df['METRIC']        == TARGET_METRIC) &
    (df['FORECAST_TYPE'] == '2025 0+12')
)
reforecast_df = (
    df[reforecast_mask][['DATE', 'METRIC_VALUE']]
    .copy()
    .assign(DATE=lambda x: pd.to_datetime(x['DATE']).dt.date)
)

# ── Build long-format prediction table ───────────────────────────────────────
actual_rows = pd.DataFrame({
    'DATE'         : test_df.index,
    'FORECAST_TYPE': 'Actual',
    'METRIC_VALUE' : test_df[TARGET_METRIC].values,
})

arimax_rows = pd.DataFrame({
    'DATE'         : forecast_mean.index,
    'FORECAST_TYPE': 'ARIMAX',
    'METRIC_VALUE' : forecast_mean.values,
})

reforecast_rows = reforecast_df.assign(FORECAST_TYPE='2025 0+12')[
    ['DATE', 'FORECAST_TYPE', 'METRIC_VALUE']
]
reforecast_rows = reforecast_rows[reforecast_rows['DATE'].isin(test_df.index)]

pred_df = pd.concat([actual_rows, arimax_rows, reforecast_rows], ignore_index=True)
pred_df['DATE'] = pred_df['DATE'].apply(
    lambda x: x.date() if isinstance(x, pd.Timestamp) else x
)

display(pred_df)

In [ ]:
def plot_results(pred_df: pd.DataFrame, metric: str) -> None:
    """Plot ARIMAX, 2025 0+12 and Actual on a single time-series chart."""
    palette = {
        'Actual'   : '#2196F3',
        'ARIMAX'   : '#FF5722',
        '2025 0+12': '#4CAF50',
    }
    order = ['Actual', 'ARIMAX', '2025 0+12']
    existing = [k for k in order if k in pred_df['FORECAST_TYPE'].unique()]

    fig, ax = plt.subplots(figsize=(20, 6))
    ax.set_title(metric, fontsize=14, fontweight='bold')

    for label in existing:
        subset = pred_df[pred_df['FORECAST_TYPE'] == label].sort_values('DATE')
        ax.plot(
            range(len(subset)),
            subset['METRIC_VALUE'].values,
            marker='o',
            label=label,
            color=palette.get(label),
            linewidth=2,
        )
        for i, (_, row) in enumerate(subset.iterrows()):
            ax.annotate(
                f"{row['METRIC_VALUE']:,.0f}",
                (i, row['METRIC_VALUE']),
                textcoords='offset points',
                xytext=(0, 8),
                ha='center',
                fontsize=7,
            )

    x_labels = (
        pred_df[pred_df['FORECAST_TYPE'] == existing[0]]
        .sort_values('DATE')['DATE']
        .astype(str)
        .tolist()
    )
    ax.set_xticks(range(len(x_labels)))
    ax.set_xticklabels(x_labels, rotation=90)
    ax.set_xlabel('DATE')
    ax.set_ylabel('METRIC_VALUE')
    ax.legend()
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.tight_layout()
    plt.show()


plot_results(pred_df, TARGET_METRIC)

In [ ]:
def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Symmetric Mean Absolute Percentage Error (%)."""
    return float(
        np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred))) * 100
    )


y_actual = (
    pred_df[pred_df['FORECAST_TYPE'] == 'Actual']
    .sort_values('DATE')
    .set_index('DATE')['METRIC_VALUE']
)

print(f"\n{'='*55}")
print(f"  Error Metrics  —  {TARGET_METRIC}")
print(f"{'='*55}")

for col in ['ARIMAX', '2025 0+12']:
    subset = pred_df[pred_df['FORECAST_TYPE'] == col]
    if subset.empty:
        print(f"\n  {col}  — no data available")
        continue

    y_pred  = subset.sort_values('DATE').set_index('DATE')['METRIC_VALUE']
    common  = y_actual.index.intersection(y_pred.index)
    if len(common) == 0:
        print(f"\n  {col}  — no overlapping dates with Actual")
        continue

    yt = y_actual[common].values.astype(float)
    yp = y_pred[common].values.astype(float)

    print(f"\n  {col}")
    print(f"    MAPE  : {mean_absolute_percentage_error(yt, yp) * 100:.2f}%")
    print(f"    SMAPE : {smape(yt, yp):.2f}%")
    print(f"    MAE   : {mean_absolute_error(yt, yp):>14,.0f}")
    print(f"    RMSE  : {np.sqrt(mean_squared_error(yt, yp)):>14,.0f}")

print(f"\n{'='*55}")